<a href="https://colab.research.google.com/github/frank-morales2020/AST/blob/main/ast_deepseek_topo_cbp_moe.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

https://huggingface.co/deepseek-ai/deepseek-moe-16b-base

## setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [3]:
!ls -lh /content/drive/MyDrive/topogpt/gguf/

total 28G
-rw------- 1 root root  17G Nov 24 10:18 deepseek-moe-16b-base-ast-aligned-nolora-Q8_0.gguf
-rw------- 1 root root 1.9G Nov 24 08:31 mmproj-voxtral-mini-4b-realtime-2602-f16.gguf
drwx------ 2 root root 4.0K Nov 21 04:59 nemotron
-rw------- 1 root root 6.5G Nov 24 07:03 voxtral-mini-4b-realtime-2602-topo-triton-bf16.gguf
-rw------- 1 root root 3.5G Nov 24 07:05 voxtral-mini-4b-realtime-2602-topo-triton-Q8_0.gguf


In [ ]:
!pip install -U bitsandbytes>=0.46.1 -q

In [ ]:
!nvidia-smi

Tue Nov 24 07:22:21 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   51C    P0             30W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install llama-cpp-python \
  --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu130 \
  --no-cache-dir

## topo-cbp

In [ ]:
import os
import re
import json
import math
import time
import shutil
import hashlib
import logging
import pathlib
import warnings

# ---------------------------------------------------------------------------
# 1. Warning Suppression & Environment Setup (Before HF / PyTorch Imports)
# ---------------------------------------------------------------------------
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning, module=r".*transformers.*")
warnings.filterwarnings("ignore", message=r".*rope_scaling.*")
warnings.filterwarnings("ignore", message=r".*AttentionMaskConverter.*")
warnings.filterwarnings("ignore", message=r".*masking_utils.*")
warnings.filterwarnings("ignore", message=r".*use_return_dict.*")
warnings.filterwarnings("ignore", message=r".*torch_dtype.*")
warnings.filterwarnings("ignore", message=r".*resume_download.*")

os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
for _offline_var in ("HF_HUB_OFFLINE", "TRANSFORMERS_OFFLINE"):
    os.environ.pop(_offline_var, None)

for _lg_name in (
    "huggingface_hub",
    "huggingface_hub.utils._http",
    "transformers",
    "transformers.configuration_utils",
    "transformers.modeling_attn_mask_utils",
):
    logging.getLogger(_lg_name).setLevel(logging.ERROR)

import torch
import torch.nn as nn
from torch.optim import Optimizer
from datasets import Dataset
from safetensors.torch import save_file
import transformers.utils.import_utils as _tf_import_utils

# Shim `is_torch_fx_available` required by remote DeepSeekMoE modeling code
if not hasattr(_tf_import_utils, "is_torch_fx_available"):
    def _is_torch_fx_available():
        try:
            import torch.fx  # noqa: F401
            return True
        except Exception:
            return False

    _tf_import_utils.is_torch_fx_available = _is_torch_fx_available

import transformers.trainer as _tf_trainer
import transformers.trainer_utils as _tf_trainer_utils
from transformers import (
    AutoConfig,
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
    TrainerCallback,
    set_seed,
)
from huggingface_hub import HfApi, hf_hub_download, snapshot_download
from huggingface_hub import constants as hf_constants

hf_constants.HF_HUB_OFFLINE = False

# Allow native PyTorch head fine-tuning on quantized models without LoRA/PEFT
_tf_trainer.validate_quantization_for_training = lambda _model: None
_tf_trainer_utils.validate_quantization_for_training = lambda _model: None


def patch_cached_deepseek_modules():
    """Patch cached modeling_deepseek.py files if they import removed is_torch_fx_available."""
    hf_modules_root = pathlib.Path(
        os.environ.get("HF_MODULES_CACHE", os.path.expanduser("~/.cache/huggingface/modules"))
    )
    if not hf_modules_root.exists():
        return
    target = "from transformers.utils.import_utils import is_torch_fx_available"
    replacement = (
        "try:\n"
        "    from transformers.utils.import_utils import is_torch_fx_available\n"
        "except ImportError:\n"
        "    is_torch_fx_available = lambda: False"
    )
    for py_file in hf_modules_root.rglob("modeling_deepseek.py"):
        txt = py_file.read_text(encoding="utf-8")
        if target in txt and replacement not in txt:
            py_file.write_text(txt.replace(target, replacement), encoding="utf-8")


def normalize_deepseek_rope_config(config):
    """Normalize rope_scaling so remote DeepSeekMoE RoPE initialization succeeds."""
    rs = getattr(config, "rope_scaling", None)
    if rs is None:
        return config
    rs_dict = dict(rs)
    raw_type = rs_dict.get("type") or rs_dict.get("rope_type")
    if raw_type in (None, "", "default"):
        config.rope_scaling = None
    else:
        rs_dict["type"] = raw_type
        if "factor" not in rs_dict or rs_dict["factor"] is None:
            rs_dict["factor"] = 1.0
        config.rope_scaling = rs_dict
    return config


# ---------------------------------------------------------------------------
# 2. Reproducibility & AST / Topo-CBP Configuration
# ---------------------------------------------------------------------------
SEED = 123
set_seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

MODEL_ID = "deepseek-ai/deepseek-moe-16b-base"
PRIME_ANCHORS = [2, 3, 5, 7, 11, 13]
AST_BLEND = 0.25
DRIFT_TOLERANCE = 0.5

drive_base = "/content/drive/MyDrive/topogpt"
run_tag = "ast-deepseek-moe-16b-base-ast-aligned-nolora"
prior_run_dir = os.path.join(drive_base, "ast-deepseek-moe-16b-base-topocbp-nolora")
output_dir = os.path.join(drive_base, run_tag)
local_work_dir = f"/tmp/{run_tag}"

os.makedirs(output_dir, exist_ok=True)
os.makedirs(local_work_dir, exist_ok=True)


def compute_euler_attenuation(anchors):
    """Compute harmonic Euler attenuation factor over prime anchors."""
    if not anchors:
        return 1.0
    inv_sum = sum(1.0 / float(p) for p in anchors)
    val = 1.0 - (0.04 * inv_sum) / (1.0 + inv_sum)
    return max(0.5, min(0.999999, val))


def compute_ast_step_scale(anchors, lambda_euler, blend=AST_BLEND):
    """Compute AST prime-kernel step scale (p ** -0.5) modulated by lambda_euler."""
    if not anchors:
        harmonic = 1.0
    else:
        harmonic = sum(float(p) ** -0.5 for p in anchors) / float(len(anchors))
    raw_scale = (harmonic / 0.45) * math.sqrt(max(lambda_euler, 1e-8))
    return (1.0 - blend) * 1.0 + blend * raw_scale


LAMBDA_EULER = compute_euler_attenuation(PRIME_ANCHORS)
AST_STEP_SCALE = compute_ast_step_scale(PRIME_ANCHORS, LAMBDA_EULER, AST_BLEND)


def count_differing_bits_i16(curr_bf16: torch.Tensor, orig_bf16: torch.Tensor) -> int:
    """Count differing bits across two bfloat16 tensors using portable 16-bit shifts."""
    xor_u16 = (curr_bf16.detach().view(torch.int16) ^ orig_bf16.detach().view(torch.int16)).to(torch.int32) & 0xFFFF
    total_bits = 0
    for shift in range(16):
        total_bits += int(((xor_u16 >> shift) & 1).sum().item())
    return total_bits


# ---------------------------------------------------------------------------
# 3. Cache & Shard Verification Helper
# ---------------------------------------------------------------------------
def ensure_complete_model_cache(model_id: str, revision: str | None = None) -> str:
    """Download and verify all model files locally without IncompleteSnapshotError."""
    api = HfApi()
    info = api.model_info(repo_id=model_id, revision=revision)
    expected_files = [
        s.rfilename
        for s in info.siblings
        if s.rfilename.endswith((".safetensors", ".bin", ".json", ".model", ".py"))
    ]

    cache_dir = snapshot_download(
        repo_id=model_id,
        revision=revision,
        local_files_only=False,
    )

    for fname in expected_files:
        local_path = os.path.join(cache_dir, fname)
        if not os.path.exists(local_path) or os.path.getsize(local_path) == 0:
            hf_hub_download(
                repo_id=model_id,
                filename=fname,
                revision=revision,
                local_dir=cache_dir,
                local_dir_use_symlinks=False,
            )

    patch_cached_deepseek_modules()
    return cache_dir


# ---------------------------------------------------------------------------
# 4. AST-Aligned TopoAdamW Optimizer (Native PyTorch, No LoRA / No PEFT)
# ---------------------------------------------------------------------------
class TopoAdamW(Optimizer):
    """
    Native PyTorch AdamW variant with:
      - Unconstrained anchor step-pressure measurement (pre_restore_l2, pre_restore_max_f32)
      - Prime-coordinate anchor protection over PRIME_ANCHORS
      - Euler attenuation (lambda_euler)
      - AST-aligned harmonic step scaling (ast_scale)
      - Memory-lean Continual Backpropagation (CBP, Nature 2024) utility tracking
      - Post-restore float32 and bit-level invariance telemetry
    """

    def __init__(
        self,
        params,
        embedding_layer: nn.Module,
        anchors=PRIME_ANCHORS,
        lr: float = 5e-5,
        betas=(0.9, 0.999),
        eps: float = 1e-8,
        weight_decay: float = 0.01,
        lambda_euler: float = LAMBDA_EULER,
        ast_scale: float = AST_STEP_SCALE,
        cbp_replacement_rate: float = 1e-4,
        cbp_maturity_threshold: int = 20,
    ):
        if lr < 0.0:
            raise ValueError(f"Invalid learning rate: {lr}")
        if eps < 0.0:
            raise ValueError(f"Invalid epsilon value: {eps}")

        defaults = dict(
            lr=lr,
            betas=betas,
            eps=eps,
            weight_decay=weight_decay,
            lambda_euler=lambda_euler,
            ast_scale=ast_scale,
            cbp_replacement_rate=cbp_replacement_rate,
            cbp_maturity_threshold=cbp_maturity_threshold,
        )
        super().__init__(params, defaults)

        self.embedding_layer = embedding_layer
        self.anchors = list(anchors)
        self.valid_anchors: list[int] = []
        self.baseline_invariant_tensor_map: dict[int, torch.Tensor] = {}
        self.last_anchor_metrics: dict[str, float | int | bool | str] = {}

        self._initialize_bedrock_anchors()

    def _initialize_bedrock_anchors(self):
        weight = self.embedding_layer.weight.data
        vocab_size = weight.shape[0]
        self.valid_anchors = [idx for idx in self.anchors if 0 <= idx < vocab_size]

        self.baseline_invariant_tensor_map.clear()
        for idx in self.valid_anchors:
            self.baseline_invariant_tensor_map[idx] = weight[idx].detach().clone().cpu()

        digest_hex = self.compute_anchor_digest()
        self.baseline_hash = digest_hex
        print(
            f"[TopoAdamW-AST] Bedrock Locked | anchors={self.valid_anchors} | "
            f"lambda_euler={self.defaults['lambda_euler']:.10f} | "
            f"ast_scale={self.defaults['ast_scale']:.6f} | Digest: {digest_hex[:16]}"
        )
        print(f"Baseline hash (seed {SEED}): {self.baseline_hash}")

    def compute_anchor_digest(self) -> str:
        hasher = hashlib.sha256()
        weight_data = self.embedding_layer.weight.data
        for idx in self.valid_anchors:
            t = weight_data[idx].detach()
            if t.dtype == torch.bfloat16:
                raw_bytes = t.view(torch.int16).cpu().numpy().tobytes()
            else:
                raw_bytes = t.float().cpu().numpy().tobytes()
            hasher.update(str(idx).encode("utf-8"))
            hasher.update(raw_bytes)
        return hasher.hexdigest()

    def _compute_unconstrained_anchor_pressure(
        self,
        emb_weight: torch.Tensor,
        grad: torch.Tensor,
        beta1: float,
        beta2: float,
        eps: float,
        weight_decay: float,
        effective_lr: float,
        step_t: int,
    ) -> tuple[float, float]:
        """
        Compute real, non-zero float32 L2 and max step delta on anchor rows
        from the unmasked gradient and current weights prior to anchor zeroing/restore.
        """
        if not self.valid_anchors:
            return 0.0, 0.0

        anchor_idx = torch.tensor(self.valid_anchors, device=emb_weight.device, dtype=torch.long)
        w_anc = emb_weight.detach()[anchor_idx].float()
        g_anc = grad.detach()[anchor_idx].float()

        # Candidate first/second moment estimates on anchor rows
        m_cand = (1.0 - beta1) * g_anc
        v_cand = (1.0 - beta2) * (g_anc * g_anc)
        bc1 = 1.0 - (beta1 ** step_t)
        bc2 = 1.0 - (beta2 ** step_t)
        step_size = effective_lr / bc1
        denom = (torch.sqrt(v_cand / bc2)).add_(eps)

        delta_f32 = -step_size * (m_cand / denom)
        if weight_decay != 0.0:
            delta_f32 = delta_f32 - (effective_lr * weight_decay) * w_anc

        l2_val = float(torch.linalg.vector_norm(delta_f32).item())
        max_val = float(delta_f32.abs().max().item())
        return l2_val, max_val

    def _restore_anchors(self):
        weight_data = self.embedding_layer.weight.data
        for idx in self.valid_anchors:
            orig = self.baseline_invariant_tensor_map[idx].to(
                device=weight_data.device, dtype=weight_data.dtype
            )
            weight_data[idx].copy_(orig)

    def measure_real_anchor_metrics(
        self,
        pre_restore_l2: float = 0.0,
        pre_restore_max_f32: float = 0.0,
    ) -> dict[str, float | int | bool | str]:
        if not self.valid_anchors:
            metrics = {
                "pre_restore_max_f32": float(pre_restore_max_f32),
                "pre_restore_l2": float(pre_restore_l2),
                "post_restore_max_f32": 0.0,
                "post_restore_l2": 0.0,
                "differing_bf16_bits": 0,
                "hash_match": True,
                "current_hash": self.baseline_hash,
            }
            self.last_anchor_metrics = metrics
            return metrics

        weight_data = self.embedding_layer.weight.data
        post_max_f32 = 0.0
        post_sq_sum = 0.0
        differing_bits = 0

        for idx in self.valid_anchors:
            orig_cpu = self.baseline_invariant_tensor_map[idx]
            curr_gpu = weight_data[idx].detach()
            orig_matched = orig_cpu.to(device=curr_gpu.device, dtype=curr_gpu.dtype)

            diff_post_f32 = (curr_gpu.float() - orig_matched.float()).abs()
            post_max_f32 = max(post_max_f32, float(diff_post_f32.max().item()))
            post_sq_sum += float((diff_post_f32 ** 2).sum().item())

            if curr_gpu.dtype == torch.bfloat16 and orig_matched.dtype == torch.bfloat16:
                differing_bits += count_differing_bits_i16(curr_gpu, orig_matched)

        current_hash = self.compute_anchor_digest()
        hash_match = current_hash == self.baseline_hash

        metrics = {
            "pre_restore_max_f32": float(pre_restore_max_f32),
            "pre_restore_l2": float(pre_restore_l2),
            "post_restore_max_f32": post_max_f32,
            "post_restore_l2": math.sqrt(post_sq_sum),
            "differing_bf16_bits": differing_bits,
            "hash_match": hash_match,
            "current_hash": current_hash,
        }
        self.last_anchor_metrics = metrics
        return metrics

    @staticmethod
    def _estimate_low_utility_threshold(util_tensor: torch.Tensor, q: float = 0.05) -> torch.Tensor:
        """Estimate low-utility quantile on a strided sample to avoid multi-GB full-tensor sorts."""
        flat = util_tensor.reshape(-1)
        stride = max(1, flat.numel() // 131072)
        sample = flat[::stride]
        k = max(1, int(sample.numel() * q))
        return torch.kthvalue(sample, k).values

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()

        emb_weight = self.embedding_layer.weight
        step_pre_l2 = 0.0
        step_pre_max_f32 = 0.0

        for group in self.param_groups:
            lr = group["lr"]
            beta1, beta2 = group["betas"]
            eps = group["eps"]
            weight_decay = group["weight_decay"]
            lambda_euler = group["lambda_euler"]
            ast_scale = group["ast_scale"]
            repl_rate = group["cbp_replacement_rate"]
            maturity_thresh = group["cbp_maturity_threshold"]

            effective_lr = lr * ast_scale

            for p in group["params"]:
                if p.grad is None:
                    continue

                grad = p.grad
                if grad.is_sparse:
                    raise RuntimeError("TopoAdamW does not support sparse gradients")

                state = self.state[p]
                if len(state) == 0:
                    state["step"] = 0
                    state["exp_avg"] = torch.zeros_like(p, memory_format=torch.preserve_format)
                    state["exp_avg_sq"] = torch.zeros_like(p, memory_format=torch.preserve_format)
                    state["cbp_age"] = torch.zeros_like(p, dtype=torch.int16)
                    state["cbp_utility"] = torch.zeros_like(p, dtype=p.dtype)

                state["step"] += 1
                step_t = state["step"]

                is_emb = p is emb_weight
                if is_emb and self.valid_anchors:
                    step_pre_l2, step_pre_max_f32 = self._compute_unconstrained_anchor_pressure(
                        emb_weight=p,
                        grad=grad,
                        beta1=beta1,
                        beta2=beta2,
                        eps=eps,
                        weight_decay=weight_decay,
                        effective_lr=effective_lr,
                        step_t=step_t,
                    )
                    grad[self.valid_anchors] = 0.0

                exp_avg = state["exp_avg"]
                exp_avg_sq = state["exp_avg_sq"]
                cbp_age = state["cbp_age"]
                cbp_utility = state["cbp_utility"]

                if is_emb:
                    grad = grad * lambda_euler

                exp_avg.mul_(beta1).add_(grad, alpha=1.0 - beta1)
                exp_avg_sq.mul_(beta2).addcmul_(grad, grad, value=1.0 - beta2)

                instant_util = (p.detach() * grad.detach()).abs()
                cbp_utility.mul_(0.9).add_(instant_util, alpha=0.1)
                del instant_util
                cbp_age.add_(1)

                bias_correction1 = 1.0 - beta1 ** step_t
                bias_correction2 = 1.0 - beta2 ** step_t
                step_size = effective_lr / bias_correction1
                denom = (exp_avg_sq.sqrt() / math.sqrt(bias_correction2)).add_(eps)

                if weight_decay != 0.0:
                    p.mul_(1.0 - effective_lr * weight_decay)

                p.addcdiv_(exp_avg, denom, value=-step_size)
                del denom

                if repl_rate > 0.0 and step_t > maturity_thresh:
                    thresh_val = self._estimate_low_utility_threshold(cbp_utility, q=0.05)
                    low_util_mask = (cbp_age > maturity_thresh) & (cbp_utility <= thresh_val)
                    if is_emb and self.valid_anchors:
                        anchor_mask = torch.zeros_like(low_util_mask, dtype=torch.bool)
                        anchor_mask[self.valid_anchors] = True
                        low_util_mask = low_util_mask & (~anchor_mask)
                        del anchor_mask
                    if low_util_mask.any():
                        exp_avg[low_util_mask] = 0.0
                        exp_avg_sq[low_util_mask] = 0.0
                        cbp_age[low_util_mask] = 0
                        cbp_utility[low_util_mask] = 0.0
                    del low_util_mask

        self._restore_anchors()
        self.measure_real_anchor_metrics(
            pre_restore_l2=step_pre_l2,
            pre_restore_max_f32=step_pre_max_f32,
        )

        return loss


# ---------------------------------------------------------------------------
# 5. Telemetry Callback & Statistical Summary Helpers
# ---------------------------------------------------------------------------
class TopoTelemetryCallback(TrainerCallback):
    def __init__(self, optimizer: TopoAdamW, log_path: str):
        self.optimizer = optimizer
        self.log_path = log_path
        self.step_records = []
        with open(self.log_path, "w", encoding="utf-8") as f:
            f.write(
                "step | pre_restore_l2 | pre_restore_max_f32 | "
                "post_restore_max_f32 | post_restore_l2 | differing_bf16_bits | "
                "hash_match | loss | grad_norm | lr\n"
            )

    def on_step_end(self, args, state, control, **kwargs):
        metrics = self.optimizer.last_anchor_metrics or self.optimizer.measure_real_anchor_metrics()
        step_num = int(state.global_step)
        total_steps = int(state.max_steps)

        print(
            f"step {step_num:3d}/{total_steps} | "
            f"pre_l2 {metrics['pre_restore_l2']:.6e} | "
            f"post_max_f32 {metrics['post_restore_max_f32']:.10f} | "
            f"post_l2 {metrics['post_restore_l2']:.10f} | "
            f"diff_bits {metrics['differing_bf16_bits']} | "
            f"hash match {metrics['hash_match']}"
        )

        record = {
            "step": step_num,
            "pre_restore_l2": float(metrics["pre_restore_l2"]),
            "pre_restore_max_f32": float(metrics["pre_restore_max_f32"]),
            "post_restore_max_f32": float(metrics["post_restore_max_f32"]),
            "post_restore_l2": float(metrics["post_restore_l2"]),
            "differing_bf16_bits": int(metrics["differing_bf16_bits"]),
            "hash_match": bool(metrics["hash_match"]),
            "loss": None,
            "grad_norm": None,
            "learning_rate": None,
        }
        self.step_records.append(record)

    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs:
            return
        if "loss" in logs and self.step_records:
            last_rec = self.step_records[-1]
            last_rec["loss"] = float(logs["loss"])
            if "grad_norm" in logs:
                last_rec["grad_norm"] = float(logs["grad_norm"])
            if "learning_rate" in logs:
                last_rec["learning_rate"] = float(logs["learning_rate"])

            with open(self.log_path, "a", encoding="utf-8") as f:
                f.write(
                    f"{last_rec['step']} | "
                    f"{last_rec['pre_restore_l2']:.8e} | "
                    f"{last_rec['pre_restore_max_f32']:.8e} | "
                    f"{last_rec['post_restore_max_f32']:.10f} | "
                    f"{last_rec['post_restore_l2']:.10f} | "
                    f"{last_rec['differing_bf16_bits']} | "
                    f"{last_rec['hash_match']} | "
                    f"{last_rec['loss']} | "
                    f"{last_rec['grad_norm']} | "
                    f"{last_rec['learning_rate']}\n"
                )


def compute_distribution_stats(values: list[float]) -> dict[str, float | int]:
    vals = [float(v) for v in values]
    n = len(vals)
    if n == 0:
        return {"count": 0, "mean": 0.0, "std": 0.0, "min": 0.0, "max": 0.0}
    mean_val = sum(vals) / n
    var_val = sum((x - mean_val) ** 2 for x in vals) / (n - 1) if n > 1 else 0.0
    return {
        "count": n,
        "mean": mean_val,
        "std": math.sqrt(var_val),
        "min": min(vals),
        "max": max(vals),
    }


def parse_prior_training_log(log_file: str):
    if not os.path.exists(log_file):
        return None
    losses = []
    post_max_vals = []
    pre_l2_vals = []
    hash_matches = []

    with open(log_file, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("step |"):
                continue
            if "|" in line:
                parts = [p.strip() for p in line.split("|")]
                if len(parts) >= 8:
                    try:
                        pre_l2_vals.append(float(parts[1]))
                        post_max_vals.append(float(parts[3]))
                        hash_matches.append(parts[6].lower() == "true")
                        if parts[7].lower() != "none":
                            losses.append(float(parts[7]))
                    except ValueError:
                        pass
            else:
                drift_m = re.search(r"drift\s+([0-9.eE+-]+)", line)
                hash_m = re.search(r"hash match\s+(True|False)", line)
                loss_m = re.search(r"'loss':\s*'([0-9.eE+-]+)'", line)
                if drift_m:
                    post_max_vals.append(float(drift_m.group(1)))
                if hash_m:
                    hash_matches.append(hash_m.group(1) == "True")
                if loss_m:
                    losses.append(float(loss_m.group(1)))

    pre_stats = compute_distribution_stats(pre_l2_vals)
    return {
        "log_file": log_file,
        "num_logged_losses": len(losses),
        "mean_train_loss": (sum(losses) / len(losses)) if losses else None,
        "max_post_restore_drift": max(post_max_vals) if post_max_vals else None,
        "pre_restore_l2_stats": pre_stats if pre_l2_vals else None,
        "all_hash_match": all(hash_matches) if hash_matches else None,
    }


def write_comparison_summary(current_records, current_final_metrics, prior_summary, out_json_path):
    curr_losses = [r["loss"] for r in current_records if r["loss"] is not None]
    pre_l2_stats = compute_distribution_stats([r["pre_restore_l2"] for r in current_records])
    pre_max_stats = compute_distribution_stats([r["pre_restore_max_f32"] for r in current_records])
    curr_post_max_f32 = [r["post_restore_max_f32"] for r in current_records]
    curr_post_l2 = [r["post_restore_l2"] for r in current_records]
    curr_diff_bits = [r["differing_bf16_bits"] for r in current_records]

    tolerance_check = {
        "drift_tolerance": DRIFT_TOLERANCE,
        "pre_restore_l2_max_within_tolerance": bool(pre_l2_stats["max"] <= DRIFT_TOLERANCE),
        "pre_restore_max_f32_max_within_tolerance": bool(pre_max_stats["max"] <= DRIFT_TOLERANCE),
        "post_restore_l2_max_within_tolerance": bool((max(curr_post_l2) if curr_post_l2 else 0.0) <= DRIFT_TOLERANCE),
        "post_restore_max_f32_max_within_tolerance": bool((max(curr_post_max_f32) if curr_post_max_f32 else 0.0) <= DRIFT_TOLERANCE),
    }

    current_summary = {
        "run_tag": run_tag,
        "model_id": MODEL_ID,
        "seed": SEED,
        "no_lora_peft": True,
        "anchors": PRIME_ANCHORS,
        "lambda_euler": LAMBDA_EULER,
        "ast_blend": AST_BLEND,
        "ast_step_scale": AST_STEP_SCALE,
        "mean_logged_train_loss": (sum(curr_losses) / len(curr_losses)) if curr_losses else None,
        "pre_restore_l2_stats": pre_l2_stats,
        "pre_restore_max_f32_stats": pre_max_stats,
        "max_post_restore_max_f32": max(curr_post_max_f32) if curr_post_max_f32 else 0.0,
        "max_post_restore_l2": max(curr_post_l2) if curr_post_l2 else 0.0,
        "total_differing_bf16_bits_post_restore": sum(curr_diff_bits),
        "tolerance_check": tolerance_check,
        "final_anchor_metrics": current_final_metrics,
    }

    delta_mean_loss = None
    if prior_summary and prior_summary["mean_train_loss"] is not None and current_summary["mean_logged_train_loss"] is not None:
        delta_mean_loss = current_summary["mean_logged_train_loss"] - prior_summary["mean_train_loss"]

    comparison = {
        "drift_tolerance": DRIFT_TOLERANCE,
        "current_run": current_summary,
        "prior_run": prior_summary,
        "delta_mean_logged_train_loss": delta_mean_loss,
    }

    with open(out_json_path, "w", encoding="utf-8") as f:
        json.dump(comparison, f, indent=2)

    return comparison


# ---------------------------------------------------------------------------
# 6. Training Execution & Artifact Export
# ---------------------------------------------------------------------------
def main():
    t_start = time.time()
    patch_cached_deepseek_modules()

    gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
    free_gb = (
        torch.cuda.mem_get_info()[0] / (1024 ** 3)
        if torch.cuda.is_available()
        else 0.0
    )
    print(f"GPU: {gpu_name} | free: {free_gb:.1f} GB")
    print(f"Loading {MODEL_ID}...")

    cache_dir = ensure_complete_model_cache(MODEL_ID)

    tokenizer = AutoTokenizer.from_pretrained(cache_dir, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    config = AutoConfig.from_pretrained(cache_dir, trust_remote_code=True)
    normalize_deepseek_rope_config(config)

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )

    model = AutoModelForCausalLM.from_pretrained(
        cache_dir,
        config=config,
        quantization_config=bnb_config,
        dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True,
    )

    trainable_names = []
    for name, param in model.named_parameters():
        if "embed_tokens.weight" in name or "lm_head.weight" in name:
            param.requires_grad = True
            param.data = param.data.to(torch.bfloat16)
            trainable_names.append(name)
        else:
            param.requires_grad = False

    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    all_params = sum(p.numel() for p in model.parameters())
    footprint_gb = model.get_memory_footprint() / (1024 ** 3)

    print(f"Model memory footprint: {footprint_gb:.1f} GB")
    print(f"Trainable (no LoRA): {trainable_names}")
    print(
        f"trainable params: {trainable_params:,} || all params: {all_params:,} || "
        f"trainable%: {100.0 * trainable_params / all_params:.4f}"
    )

    optimizer = TopoAdamW(
        params=[p for p in model.parameters() if p.requires_grad],
        embedding_layer=model.model.embed_tokens,
        anchors=PRIME_ANCHORS,
        lr=5e-5,
        weight_decay=0.01,
        lambda_euler=LAMBDA_EULER,
        ast_scale=AST_STEP_SCALE,
    )

    dialogues = [
        "User: How does arithmetic spectral theory stabilize continual learning?\nAssistant: Prime-coordinate anchors constrain invariant subspace drift while preserving adaptation plasticity.",
        "User: Why avoid LoRA in deterministic continual backpropagation?\nAssistant: Direct native PyTorch updates on designated heads allow explicit invariant enforcement and auditability.",
        "User: What role does Euler attenuation play in TopoAdamW?\nAssistant: It smooths embedding step magnitudes across prime-indexed spectral coordinates.",
        "User: How is anchor integrity verified during training?\nAssistant: Pre- and post-restore float32 norms, bit checks, and SHA-256 digests are recorded step by step.",
        "User: How does CBP complement topological anchoring?\nAssistant: Continual backpropagation tracks utility and age to prevent feature stagnation over streaming updates.",
        "User: Why evaluate on DeepSeekMoE-16B-Base?\nAssistant: Mixture-of-experts architectures test stability across sparse routing and shared representations.",
        "User: Can we distinguish zero numeric difference from mathematical zero?\nAssistant: Yes, exact float32/bit-level invariance under a fixed representation should be reported explicitly.",
        "User: How does harmonic step scaling interact with learning rate?\nAssistant: A blended prime-harmonic factor scales step magnitudes smoothly alongside the LR schedule.",
        "User: Why log both pre-restore and post-restore norms?\nAssistant: Pre-restore norm shows attempted update magnitude; post-restore norm confirms exact invariant recovery.",
        "User: How should results be organized for manuscript comparison?\nAssistant: Keep identical seeds, budgets, and directory artifacts so runs compare cleanly side by side.",
    ]

    raw_ds = Dataset.from_dict({"text": dialogues})

    def tokenize_fn(batch):
        enc = tokenizer(
            batch["text"],
            truncation=True,
            max_length=256,
            padding="max_length",
        )
        enc["labels"] = enc["input_ids"].copy()
        return enc

    tokenized_ds = raw_ds.map(tokenize_fn, batched=True, remove_columns=["text"])
    data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

    log_path = os.path.join(local_work_dir, "training_log.txt")
    telemetry_cb = TopoTelemetryCallback(optimizer=optimizer, log_path=log_path)

    training_args = TrainingArguments(
        output_dir=local_work_dir,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        num_train_epochs=15,
        learning_rate=5e-5,
        lr_scheduler_type="cosine",
        warmup_steps=4,
        logging_steps=1,
        save_strategy="no",
        bf16=True,
        seed=SEED,
        report_to="none",
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_ds,
        data_collator=data_collator,
        optimizers=(optimizer, None),
        callbacks=[telemetry_cb],
    )

    print(f"LR: {training_args.learning_rate} | Total Steps: {trainer.state.max_steps} | Warmup Steps: {training_args.warmup_steps}")
    print("Starting AST-aligned Topo-CBP training (no LoRA) on DeepSeekMoE-16B-Base via HF Trainer...")

    train_result = trainer.train()
    final_metrics = optimizer.measure_real_anchor_metrics(
        pre_restore_l2=optimizer.last_anchor_metrics.get("pre_restore_l2", 0.0),
        pre_restore_max_f32=optimizer.last_anchor_metrics.get("pre_restore_max_f32", 0.0),
    )

    pre_l2_stats = compute_distribution_stats([r["pre_restore_l2"] for r in telemetry_cb.step_records])
    pre_max_stats = compute_distribution_stats([r["pre_restore_max_f32"] for r in telemetry_cb.step_records])

    logged_losses = [r["loss"] for r in telemetry_cb.step_records if r["loss"] is not None]
    mean_logged_loss = (sum(logged_losses) / len(logged_losses)) if logged_losses else float(train_result.training_loss)

    runtime_sec = time.time() - t_start
    print(f"{{'train_runtime': '{runtime_sec:.1f}', 'train_loss': '{mean_logged_loss:.4f}', 'epoch': '{training_args.num_train_epochs}'}}")
    print("=" * 70)
    print(f"Drift Tolerance:                     {DRIFT_TOLERANCE}")
    print(f"Pre-Restore L2 (mean ± std):         {pre_l2_stats['mean']:.8e} ± {pre_l2_stats['std']:.8e}")
    print(f"Pre-Restore L2 (min / max):          {pre_l2_stats['min']:.8e} / {pre_l2_stats['max']:.8e}")
    print(f"Pre-Restore Max f32 (mean ± std):    {pre_max_stats['mean']:.8e} ± {pre_max_stats['std']:.8e}")
    print(f"Post-Restore Max f32 Drift:          {final_metrics['post_restore_max_f32']:.10f}")
    print(f"Post-Restore L2 Drift:               {final_metrics['post_restore_l2']:.10f}")
    print(f"Post-Restore Differing bf16 Bits:    {final_metrics['differing_bf16_bits']}")
    print(f"Hash match (certified digest):       {final_metrics['hash_match']}")
    print(f"Final hash:                          {final_metrics['current_hash']}")
    print(f"Mean logged train loss:              {mean_logged_loss:.6f}")
    print("=" * 70)

    within_tol = (
        pre_l2_stats["max"] <= DRIFT_TOLERANCE
        and pre_max_stats["max"] <= DRIFT_TOLERANCE
        and final_metrics["post_restore_l2"] <= DRIFT_TOLERANCE
    )
    integrity_pass = bool(final_metrics["hash_match"]) and final_metrics["differing_bf16_bits"] == 0 and within_tol
    print(
        f"Training Complete. Pre-Restore L2 Mean: {pre_l2_stats['mean']:.6e} "
        f"(Tolerance: {DRIFT_TOLERANCE}, Within Tol: {within_tol})"
    )
    print(f"INTEGRITY VERDICT: {'PASS' if integrity_pass else 'WARN'}")

    trainable_state = {
        k: v.detach().cpu().contiguous()
        for k, v in model.state_dict().items()
        if any(tag in k for tag in ("embed_tokens.weight", "lm_head.weight"))
    }
    safetensors_path = os.path.join(local_work_dir, "topo_cbp_trained_weights.safetensors")
    save_file(trainable_state, safetensors_path)

    tokenizer.save_pretrained(local_work_dir)

    prior_log_candidates = [
        os.path.join(prior_run_dir, "training_log.txt"),
        os.path.join(drive_base, "deepseek-moe-16b-base-topocbp-nolora", "training_log.txt"),
    ]
    prior_summary = None
    for cand in prior_log_candidates:
        if os.path.exists(cand):
            prior_summary = parse_prior_training_log(cand)
            break

    comp_json_local = os.path.join(local_work_dir, "comparison_summary.json")
    comparison = write_comparison_summary(
        current_records=telemetry_cb.step_records,
        current_final_metrics=final_metrics,
        prior_summary=prior_summary,
        out_json_path=comp_json_local,
    )

    print("--- Side-by-Side Comparison vs. Prior Run ---")
    print(json.dumps(comparison, indent=2))

    print("Copying outputs to Google Drive...")
    for fname in os.listdir(local_work_dir):
        src_file = os.path.join(local_work_dir, fname)
        dst_file = os.path.join(output_dir, fname)
        if os.path.isfile(src_file):
            shutil.copy2(src_file, dst_file)
        elif os.path.isdir(src_file):
            if os.path.exists(dst_file):
                shutil.rmtree(dst_file)
            shutil.copytree(src_file, dst_file)

    print(f"Saved to {output_dir}")


if __name__ == "__main__":
    main()

GPU: NVIDIA L4 | free: 21.8 GB
Loading deepseek-ai/deepseek-moe-16b-base...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 16 files:   0%|          | 0/16 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/5466 [00:00<?, ?it/s]

Model memory footprint: 8.3 GB
Trainable (no LoRA): ['model.embed_tokens.weight', 'lm_head.weight']
trainable params: 419,430,400 || all params: 8,399,407,104 || trainable%: 4.9936
[TopoAdamW-AST] Bedrock Locked | anchors=[2, 3, 5, 7, 11, 13] | lambda_euler=0.9770646816 | ast_scale=0.996064 | Digest: b6fa0d3775fbcf6c
Baseline hash (seed 123): b6fa0d3775fbcf6c494a1ae229170ef35b1e4f62406a4080978a504b09d5a8af


Map:   0%|          | 0/10 [00:00<?, ? examples/s]

LR: 5e-05 | Total Steps: 0 | Warmup Steps: 4
Starting AST-aligned Topo-CBP training (no LoRA) on DeepSeekMoE-16B-Base via HF Trainer...
step   1/45 | pre_l2 0.000000e+00 | post_max_f32 0.0000000000 | post_l2 0.0000000000 | diff_bits 0 | hash match True
{'loss': '7.596', 'grad_norm': '12.62', 'learning_rate': '0', 'epoch': '0.4'}
step   2/45 | pre_l2 5.917754e-04 | post_max_f32 0.0000000000 | post_l2 0.0000000000 | diff_bits 0 | hash match True
{'loss': '7.022', 'grad_norm': '12.06', 'learning_rate': '1.25e-05', 'epoch': '0.8'}
step   3/45 | pre_l2 1.013172e-03 | post_max_f32 0.0000000000 | post_l2 0.0000000000 | diff_bits 0 | hash match True
{'loss': '7.304', 'grad_norm': '13.69', 'learning_rate': '2.5e-05', 'epoch': '1'}
step   4/45 | pre_l2 1.385793e-03 | post_max_f32 0.0000000000 | post_l2 0.0000000000 | diff_bits 0 | hash match True
{'loss': '7.106', 'grad_norm': '11.19', 'learning_rate': '3.75e-05', 'epoch': '1.4'}
step   5/45 | pre_l2 1.732872e-03 | post_max_f32 0.0000000000 | po

In [ ]:
!ls -lh /content/drive/MyDrive/topogpt/ast-deepseek-moe-16b-base-ast-aligned-nolora/

total 808M
-rw------- 1 root root 1.9K Nov 24 07:37 comparison_summary.json
-rw------- 1 root root  395 Nov 24 07:37 tokenizer_config.json
-rw------- 1 root root 7.2M Nov 24 07:37 tokenizer.json
-rw------- 1 root root 801M Nov 24 07:37 topo_cbp_trained_weights.safetensors
-rw------- 1 root root 5.9K Nov 24 07:37 training_log.txt


## inference-local

In [ ]:
import os
import json
import math
import hashlib
import torch
from safetensors.torch import load_file

RUN_DIR = "/content/drive/MyDrive/topogpt/ast-deepseek-moe-16b-base-ast-aligned-nolora"
PRIME_ANCHORS = [2, 3, 5, 7, 11, 13]
DRIFT_TOLERANCE = 0.5


def anchor_digest_from_tensor(emb_weight: torch.Tensor, anchors: list[int]) -> str:
    hasher = hashlib.sha256()
    vocab_size = emb_weight.shape[0]
    for idx in anchors:
        if 0 <= idx < vocab_size:
            row = emb_weight[idx].detach()
            raw_bytes = (
                row.view(torch.int16).cpu().numpy().tobytes()
                if row.dtype == torch.bfloat16
                else row.float().cpu().numpy().tobytes()
            )
            hasher.update(str(idx).encode("utf-8"))
            hasher.update(raw_bytes)
    return hasher.hexdigest()


def parse_log_pre_l2(log_path: str) -> list[float]:
    vals = []
    with open(log_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("step |"):
                continue
            parts = [p.strip() for p in line.split("|")]
            if len(parts) >= 2:
                vals.append(float(parts[1]))
    return vals


def mean_std(vals: list[float]) -> tuple[float, float]:
    n = len(vals)
    if n == 0:
        return 0.0, 0.0
    m = sum(vals) / n
    var = sum((x - m) ** 2 for x in vals) / (n - 1) if n > 1 else 0.0
    return m, math.sqrt(var)


def main():
    weights_path = os.path.join(RUN_DIR, "topo_cbp_trained_weights.safetensors")
    summary_path = os.path.join(RUN_DIR, "comparison_summary.json")
    log_path = os.path.join(RUN_DIR, "training_log.txt")

    for p in (weights_path, summary_path, log_path):
        assert os.path.exists(p), f"Missing artifact: {p}"

    state = load_file(weights_path, device="cpu")
    with open(summary_path, "r", encoding="utf-8") as f:
        summary = json.load(f)

    emb_key = next((k for k in state if "embed_tokens.weight" in k), None)
    assert emb_key is not None, f"embed_tokens.weight not found in keys: {list(state.keys())}"

    reloaded_digest = anchor_digest_from_tensor(state[emb_key], PRIME_ANCHORS)
    curr_run = summary.get("current_run", {})
    expected_digest = curr_run.get("final_anchor_metrics", {}).get("current_hash")

    log_pre_l2 = parse_log_pre_l2(log_path)
    calc_mean, calc_std = mean_std(log_pre_l2)
    json_stats = curr_run.get("pre_restore_l2_stats", {})

    max_pre_l2 = max(log_pre_l2) if log_pre_l2 else 0.0
    within_tol = max_pre_l2 <= DRIFT_TOLERANCE
    digest_matches_json = (expected_digest is None) or (reloaded_digest == expected_digest)

    print(f"Loaded keys:            {list(state.keys())} (dtype={state[emb_key].dtype})")
    print(f"Reloaded anchor digest: {reloaded_digest}")
    print(f"JSON digest match:      {digest_matches_json}")
    print(f"Log step count:         {len(log_pre_l2)}")
    print(f"Pre-Restore L2 (calc):  {calc_mean:.8e} ± {calc_std:.8e} (max={max_pre_l2:.8e})")
    print(f"Pre-Restore L2 (JSON):  {json_stats.get('mean', 0.0):.8e} ± {json_stats.get('std', 0.0):.8e}")
    print(f"Drift Tolerance ({DRIFT_TOLERANCE}):  within_tol={within_tol}")


if __name__ == "__main__":
    main()

Loaded keys:            ['lm_head.weight', 'model.embed_tokens.weight'] (dtype=torch.bfloat16)
Reloaded anchor digest: b6fa0d3775fbcf6c494a1ae229170ef35b1e4f62406a4080978a504b09d5a8af
JSON digest match:      True
Log step count:         45
Pre-Restore L2 (calc):  7.81915706e-04 ± 5.27501950e-04 (max=1.73287175e-03)
Pre-Restore L2 (JSON):  7.81915706e-04 ± 5.27501950e-04
Drift Tolerance (0.5):  within_tol=True


In [ ]:
!ls -ltha /content/drive/MyDrive/topogpt/llama.cpp/; ls /content/drive/MyDrive/topogpt/llama.cpp/build/bin/

## merge

In [ ]:
import os
import json
import shutil
from huggingface_hub import snapshot_download
from safetensors.torch import load_file, save_file

MODEL_ID = "deepseek-ai/deepseek-moe-16b-base"
RUN_DIR = "/content/drive/MyDrive/topogpt/ast-deepseek-moe-16b-base-ast-aligned-nolora"
TRAINED_WEIGHTS = os.path.join(RUN_DIR, "topo_cbp_trained_weights.safetensors")
MERGED_HF_DIR = "/tmp/deepseek-moe-16b-ast-merged-hf"

assert os.path.isfile(TRAINED_WEIGHTS), f"Missing trained weights: {TRAINED_WEIGHTS}"

cache_dir = snapshot_download(repo_id=MODEL_ID, local_files_only=True)
index_path = os.path.join(cache_dir, "model.safetensors.index.json")
assert os.path.isfile(index_path), f"Shard index not found: {index_path}"

with open(index_path, "r", encoding="utf-8") as f:
    index_data = json.load(f)
weight_map = index_data["weight_map"]

trained_state = load_file(TRAINED_WEIGHTS, device="cpu")
affected_shards = {}
for key, tensor in trained_state.items():
    assert key in weight_map, f"Key '{key}' not found in model.safetensors.index.json"
    shard_name = weight_map[key]
    affected_shards.setdefault(shard_name, {})[key] = tensor

if os.path.exists(MERGED_HF_DIR):
    shutil.rmtree(MERGED_HF_DIR)
os.makedirs(MERGED_HF_DIR, exist_ok=True)

for fname in sorted(os.listdir(cache_dir)):
    src = os.path.realpath(os.path.join(cache_dir, fname))
    dst = os.path.join(MERGED_HF_DIR, fname)

    if not os.path.isfile(src):
        continue

    if fname in affected_shards:
        shard_tensors = load_file(src, device="cpu")
        for key, new_tensor in affected_shards[fname].items():
            shard_tensors[key] = new_tensor.to(dtype=shard_tensors[key].dtype).contiguous()
        save_file(shard_tensors, dst, metadata={"format": "pt"})
        print(f"Patched shard: {fname}")
    else:
        shutil.copy2(src, dst)
        print(f"Copied file:   {fname}")

dst_index = os.path.join(MERGED_HF_DIR, "model.safetensors.index.json")
with open(dst_index, "w", encoding="utf-8") as f:
    json.dump(index_data, f, indent=2)

# Verify zero symlinks remain inside MERGED_HF_DIR
symlinks = [
    fn for fn in os.listdir(MERGED_HF_DIR)
    if os.path.islink(os.path.join(MERGED_HF_DIR, fn))
]
assert not symlinks, f"Found unexpected symlinks: {symlinks}"

total_gb = sum(
    os.path.getsize(os.path.join(MERGED_HF_DIR, fn))
    for fn in os.listdir(MERGED_HF_DIR)
    if os.path.isfile(os.path.join(MERGED_HF_DIR, fn))
) / (1024 ** 3)

print(f"Clean merge ready at {MERGED_HF_DIR} (0 symlinks, {total_gb:.2f} GB total)")

Copied file:   .gitattributes
Copied file:   README.md
Copied file:   config.json
Copied file:   configuration_deepseek.py
Copied file:   generation_config.json
Patched shard: model-00001-of-00007.safetensors
Copied file:   model-00002-of-00007.safetensors
Copied file:   model-00003-of-00007.safetensors
Copied file:   model-00004-of-00007.safetensors
Copied file:   model-00005-of-00007.safetensors
Copied file:   model-00006-of-00007.safetensors
Patched shard: model-00007-of-00007.safetensors
Copied file:   model.safetensors.index.json
Copied file:   modeling_deepseek.py
Copied file:   tokenizer.json
Copied file:   tokenizer_config.json
Clean merge ready at /tmp/deepseek-moe-16b-ast-merged-hf (0 symlinks, 30.51 GB total)


In [ ]:
!ls -ltha /tmp/deepseek-moe-16b-ast-merged-hf/

ls: cannot access '/tmp/deepseek-moe-16b-ast-merged-hf/': No such file or directory


In [ ]:
!rm -rf /tmp/deepseek-moe-16b-ast-merged-hf/

## gguf

In [ ]:
import os
import subprocess

LLAMA_CPP_DIR = "/content/drive/MyDrive/topogpt/llama.cpp"
MERGED = "/tmp/deepseek-moe-16b-ast-merged-hf"
GGUF_OUT_DIR = "/content/drive/MyDrive/topogpt/gguf"
GGUF_BF16 = os.path.join(GGUF_OUT_DIR, "deepseek-moe-16b-base-ast-aligned-nolora-BF16.gguf")

if not os.path.isdir(MERGED):
    raise FileNotFoundError(f"Merged HF directory not found at {MERGED}. Run the merge step first.")

os.makedirs(GGUF_OUT_DIR, exist_ok=True)

cmd = [
    "python",
    f"{LLAMA_CPP_DIR}/convert_hf_to_gguf.py",
    MERGED,
    "--outfile",
    GGUF_BF16,
    "--outtype",
    "bf16",
]

proc = subprocess.run(cmd, input="y\n", capture_output=True, text=True)
if proc.returncode != 0:
    print("STDOUT:\n", proc.stdout[-4000:])
    print("STDERR:\n", proc.stderr[-4000:])
    raise RuntimeError(f"convert_hf_to_gguf.py failed with exit code {proc.returncode}")

print(GGUF_BF16, f"{os.path.getsize(GGUF_BF16) / 1e9:.1f} GB")

/content/drive/MyDrive/topogpt/gguf/deepseek-moe-16b-base-ast-aligned-nolora-BF16.gguf 32.8 GB


## gguf-quantize

In [ ]:
!chmod a+x /content/drive/MyDrive/topogpt/llama.cpp/build/bin/llama-quantize
!ls -ltha /content/drive/MyDrive/topogpt/llama.cpp/build/bin/llama-quantize

-rwx------ 1 root root 18K Nov  2 10:56 /content/drive/MyDrive/topogpt/llama.cpp/build/bin/llama-quantize


In [ ]:
import os
import subprocess

LLAMA_CPP_DIR = "/content/drive/MyDrive/topogpt/llama.cpp"
BUILD_DIR = os.path.join(LLAMA_CPP_DIR, "build")
GGUF_OUT_DIR = "/content/drive/MyDrive/topogpt/gguf"
GGUF_BF16 = os.path.join(GGUF_OUT_DIR, "deepseek-moe-16b-base-ast-aligned-nolora-BF16.gguf")
GGUF_Q8_0 = os.path.join(GGUF_OUT_DIR, "deepseek-moe-16b-base-ast-aligned-nolora-Q8_0.gguf")

if not os.path.isfile(GGUF_BF16):
    raise FileNotFoundError(f"BF16 GGUF not found at {GGUF_BF16}. Run the BF16 conversion step first.")

lib_dirs = [
    os.path.join(BUILD_DIR, "bin"),
    os.path.join(BUILD_DIR, "src"),
    os.path.join(BUILD_DIR, "ggml", "src"),
    BUILD_DIR,
]
existing_lib_dirs = [p for p in lib_dirs if os.path.isdir(p)]

env = os.environ.copy()
prev_ld = env.get("LD_LIBRARY_PATH", "")
env["LD_LIBRARY_PATH"] = ":".join(existing_lib_dirs + ([prev_ld] if prev_ld else []))

quantize_bin = os.path.join(BUILD_DIR, "bin", "llama-quantize")
if not os.path.isfile(quantize_bin):
    subprocess.run(
        ["cmake", "-B", BUILD_DIR, "-S", LLAMA_CPP_DIR, "-DGGML_CUDA=OFF"],
        env=env,
        check=True,
    )
    subprocess.run(
        ["cmake", "--build", BUILD_DIR, "--target", "llama-quantize", "-j"],
        env=env,
        check=True,
    )

cmd = [quantize_bin, GGUF_BF16, GGUF_Q8_0, "Q8_0"]
proc = subprocess.run(cmd, env=env, capture_output=True, text=True)
if proc.returncode != 0:
    print("STDOUT:\n", proc.stdout[-4000:])
    print("STDERR:\n", proc.stderr[-4000:])
    raise RuntimeError(f"llama-quantize failed with exit code {proc.returncode}")

print(GGUF_Q8_0, f"{os.path.getsize(GGUF_Q8_0) / 1e9:.1f} GB")

/content/drive/MyDrive/topogpt/gguf/deepseek-moe-16b-base-ast-aligned-nolora-Q8_0.gguf 17.4 GB


## gguf-evaluator

In [ ]:
import os
from gguf import GGUFReader

GGUF_PATH = "/content/drive/MyDrive/topogpt/gguf/deepseek-moe-16b-base-ast-aligned-nolora-Q8_0.gguf"

assert os.path.isfile(GGUF_PATH), f"File not found: {GGUF_PATH}"
file_size = os.path.getsize(GGUF_PATH)
print(f"File: {GGUF_PATH} ({file_size / 1e9:.2f} GB)")

reader = GGUFReader(GGUF_PATH, "r")
print(f"Tensor count: {len(reader.tensors)}")

interesting_keys = [
    "general.architecture",
    "general.name",
    "general.quantization_version",
    "tokenizer.ggml.pre",
]
for key in interesting_keys:
    field = reader.fields.get(key)
    if field is not None:
        val = field.parts[field.data[0]]
        if hasattr(val, "tobytes"):
            val = val.tobytes().decode("utf-8", errors="replace")
        print(f"  {key}: {val}")

# Verify every tensor's offset + byte length stays within file_size
max_end = 0
for t in reader.tensors:
    end = int(t.data_offset) + int(t.n_bytes)
    if end > max_end:
        max_end = end
    assert end <= file_size, f"Tensor '{t.name}' exceeds file size ({end} > {file_size})"

print(f"Integrity OK: all {len(reader.tensors)} tensors fit within {file_size} bytes (max byte offset: {max_end})")

File: /content/drive/MyDrive/topogpt/gguf/deepseek-moe-16b-base-ast-aligned-nolora-Q8_0.gguf (17.41 GB)
Tensor count: 363
  general.architecture: deepseek
  general.name: Deepseek Moe 16b Ast Merged Hf
  general.quantization_version:    
  tokenizer.ggml.pre: deepseek-llm
Integrity OK: all 363 tensors fit within 17413944128 bytes (max byte offset: 17413944128)


In [7]:
import os
from llama_cpp import Llama

MODEL_PATH = "/content/drive/MyDrive/topogpt/gguf/deepseek-moe-16b-base-ast-aligned-nolora-Q8_0.gguf"
assert os.path.isfile(MODEL_PATH), f"Model not found: {MODEL_PATH}"

print("Loading model...")
llm = Llama(
    model_path=MODEL_PATH,
    n_gpu_layers=-1,  # Try 0 if GPU offloading has issues with this MoE GGUF
    n_ctx=512,
    seed=123,
    verbose=False,
)
print("Model loaded. Running prompt...")

prompt = "What is topological continual backpropagation?"
output = llm(
    prompt,
    max_tokens=128,
    temperature=0.0,
)

print("Raw choices:", output.get("choices"))
text = output["choices"][0]["text"]
print("Generated text:\n" + repr(text))

Loading model...
Model loaded. Running prompt...
Raw choices: [{'text': '\n\nTopological continual backpropagation (TCB) is a method for training deep neural networks that is based on the idea of topological data analysis. It was first proposed by researchers at the University of California, Berkeley in 2017 and has since been used to train a variety of deep neural networks.\n\nTCB works by using a topological representation of the data to train the network. This representation is based on the idea of persistent homology, which is a mathematical tool for studying the topology of data. By using persistent homology, TCB is able to capture the global structure of the data and use it to train the network', 'index': 0, 'logprobs': None, 'finish_reason': 'length'}]
Generated text:
'\n\nTopological continual backpropagation (TCB) is a method for training deep neural networks that is based on the idea of topological data analysis. It was first proposed by researchers at the University of Calif

In [9]:
import os
import hashlib
from llama_cpp import Llama

MODEL_PATH = "/content/drive/MyDrive/topogpt/gguf/deepseek-moe-16b-base-ast-aligned-nolora-Q8_0.gguf"
assert os.path.isfile(MODEL_PATH), f"Model not found: {MODEL_PATH}"

llm = Llama(
    model_path=MODEL_PATH,
    n_gpu_layers=-1,
    n_ctx=1024,
    seed=123,
    verbose=False,
)

prompts = [
    "Topological continual backpropagation preserves prior knowledge by",
    "In arithmetic spectral theory, the LEFM operator connects prime distributions to",
    "Deterministic model verification without adapters or LoRA ensures that",
]


def run_pass(prompts_list):
    hashes = []
    outputs = []
    for prompt in prompts_list:
        llm.set_seed(123)
        out = llm(
            prompt,
            max_tokens=128,
            temperature=0.0,
            repeat_penalty=1.1,
        )
        text = out["choices"][0]["text"]
        digest = hashlib.sha256(text.encode("utf-8")).hexdigest()
        hashes.append(digest)
        outputs.append(text)
    return hashes, outputs


pass1_hashes, pass1_outputs = run_pass(prompts)
pass2_hashes, pass2_outputs = run_pass(prompts)

for idx, prompt in enumerate(prompts, start=1):
    h1, h2 = pass1_hashes[idx - 1], pass2_hashes[idx - 1]
    assert h1 == h2, f"Mismatch on prompt [{idx}]: {h1} != {h2}"
    print(f"[{idx}] MATCH OK | SHA-256: {h1}")
    print(f"    Prompt: {prompt}")
    print(f"    Output: {pass1_outputs[idx - 1].strip()!r}\n")

print(f"All {len(prompts)} prompts verified identical across both passes.")

[1] MATCH OK | SHA-256: 1c21021c0ba6ee691645183518b7dadc5292e657b7a3edd325e589638b321cdf
    Prompt: Topological continual backpropagation preserves prior knowledge by
    Output: 'learning a continuous function from the input to the output.\n\n## 2 Related Work\n\n**Topological Backpropagation:** Topological backpropagation [<CITATION>] is an extension of topological data analysis (TDA) that uses persistent homology to extract topological features and then applies them as filters in deep neural networks. The authors show that this approach can be used for classification, regression, and anomaly detection tasks.\n\n**Topological Deep Learning:** Topological deep learning [<CITATION>] is a framework for designing deep architectures with topological properties such as persistence homology. It has been applied to various problems including image segmentation, object'

[2] MATCH OK | SHA-256: 9790733a6960e6ec60bc2f06be8aebd3b6b2b1670aa400722eaae4cb9f512002
    Prompt: In arithmetic spectra

## hf

In [10]:
import os
from google.colab import userdata
from huggingface_hub import HfApi, create_repo

# Retrieve your token from Colab Secrets
HF_TOKEN = userdata.get('HF_TOKEN')
assert HF_TOKEN, "HF_TOKEN not found in Colab Secrets."

user_or_name = 'frankmorales2020'
REPO_ID = f"{user_or_name}/deepseek-moe-16b-base-ast-aligned-nolora-GGUF"

GGUF_DIR = "/content/drive/MyDrive/topogpt/gguf"
FILENAME = "deepseek-moe-16b-base-ast-aligned-nolora-Q8_0.gguf"
local_path = os.path.join(GGUF_DIR, FILENAME)
assert os.path.isfile(local_path), f"Missing file: {local_path}"

api = HfApi(token=HF_TOKEN)
create_repo(
    repo_id=REPO_ID,
    repo_type="model",
    exist_ok=True,
    token=HF_TOKEN,
)

print(f"Uploading {FILENAME} ({os.path.getsize(local_path) / 1e9:.2f} GB) to {REPO_ID}...")
api.upload_file(
    path_or_fileobj=local_path,
    path_in_repo=FILENAME,
    repo_id=REPO_ID,
    repo_type="model",
    commit_message=f"Upload {FILENAME}",
)

print(f"Done. Uploaded {FILENAME} to https://huggingface.co/{REPO_ID}")

Uploading deepseek-moe-16b-base-ast-aligned-nolora-Q8_0.gguf (17.41 GB) to frankmorales2020/deepseek-moe-16b-base-ast-aligned-nolora-GGUF...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...-aligned-nolora-Q8_0.gguf:   0%|          |  540kB / 17.4GB            

Validating                    :           |   0%            

Done. Uploaded deepseek-moe-16b-base-ast-aligned-nolora-Q8_0.gguf to https://huggingface.co/frankmorales2020/deepseek-moe-16b-base-ast-aligned-nolora-GGUF


## inference

In [12]:
import os
import re
import json
import math
import time
import shutil
import hashlib
import logging
import pathlib
import warnings

# ---------------------------------------------------------------------------
# 1. Warning Suppression & Environment Setup (Before HF / PyTorch Imports)
# ---------------------------------------------------------------------------
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning, module=r".*transformers.*")
warnings.filterwarnings("ignore", message=r".*rope_scaling.*")
warnings.filterwarnings("ignore", message=r".*AttentionMaskConverter.*")
warnings.filterwarnings("ignore", message=r".*masking_utils.*")
warnings.filterwarnings("ignore", message=r".*use_return_dict.*")
warnings.filterwarnings("ignore", message=r".*torch_dtype.*")
warnings.filterwarnings("ignore", message=r".*resume_download.*")

os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
for _offline_var in ("HF_HUB_OFFLINE", "TRANSFORMERS_OFFLINE"):
    os.environ.pop(_offline_var, None)

for _lg_name in (
    "huggingface_hub",
    "huggingface_hub.utils._http",
    "transformers",
    "transformers.configuration_utils",
    "transformers.modeling_attn_mask_utils",
):
    logging.getLogger(_lg_name).setLevel(logging.ERROR)

In [14]:
import hashlib
from google.colab import userdata
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

# Retrieve your token from Colab Secrets
HF_TOKEN = userdata.get('HF_TOKEN')

user_or_name = 'frankmorales2020'
REPO_ID = f"{user_or_name}/deepseek-moe-16b-base-ast-aligned-nolora-GGUF"
FILENAME = "deepseek-moe-16b-base-ast-aligned-nolora-Q8_0.gguf"

local_gguf_path = hf_hub_download(
    repo_id=REPO_ID,
    filename=FILENAME,
    repo_type="model",
    token=HF_TOKEN,
    local_dir="/tmp/hf_gguf_test",
)
print(f"Downloaded GGUF to: {local_gguf_path}")

llm = Llama(
    model_path=local_gguf_path,
    n_gpu_layers=-1,
    n_ctx=1024,
    seed=123,
    verbose=False,
)

test_prompts = [
    "Topological continual backpropagation preserves prior knowledge by",
    "In arithmetic spectral theory, the LEFM operator connects prime distributions to",
    "Deterministic model verification without adapters or LoRA ensures that",
]

for idx, prompt in enumerate(test_prompts, start=1):
    llm.set_seed(123)
    result = llm(
        prompt,
        max_tokens=128,
        temperature=0.0,
        repeat_penalty=1.1,
    )
    text = result["choices"][0]["text"]
    digest = hashlib.sha256(text.encode("utf-8")).hexdigest()

    print(f"[{idx}] SHA-256: {digest}")
    print(f"    Prompt: {prompt}")
    print(f"    Output: {text.strip()!r}\n")

deepseek-moe-16b-base-ast-aligned-nolora(…): reconstructing file:   0%|          |  0.00B / 17.4GB            

deepseek-moe-16b-base-ast-aligned-nolora(…): downloading bytes:           |  0.00B            

Downloaded GGUF to: /tmp/hf_gguf_test/deepseek-moe-16b-base-ast-aligned-nolora-Q8_0.gguf
[1] SHA-256: 1c21021c0ba6ee691645183518b7dadc5292e657b7a3edd325e589638b321cdf
    Prompt: Topological continual backpropagation preserves prior knowledge by
    Output: 'learning a continuous function from the input to the output.\n\n## 2 Related Work\n\n**Topological Backpropagation:** Topological backpropagation [<CITATION>] is an extension of topological data analysis (TDA) that uses persistent homology to extract topological features and then applies them as filters in deep neural networks. The authors show that this approach can be used for classification, regression, and anomaly detection tasks.\n\n**Topological Deep Learning:** Topological deep learning [<CITATION>] is a framework for designing deep architectures with topological properties such as persistence homology. It has been applied to various problems including image segmentation, object'

[2] SHA-256: 9790733a6960e6ec60bc2f06be8aebd